In [0]:
bronze_df = (
    spark.read
    .option("multiLine", "true")
    .json("/Volumes/workspace/default/weather_raw")
)

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_bronze")

In [0]:
from pyspark.sql.functions import (
    col,
    from_unixtime,
    to_timestamp
)

silver_df = bronze_df.select(
    col("name").alias("city"),
    col("sys.country").alias("country"),

    col("main.temp").alias("temperature"),
    col("main.feels_like").alias("feels_like"),

    col("main.temp_min").alias("temp_min"),
    col("main.temp_max").alias("temp_max"),

    col("main.humidity").alias("humidity"),
    col("main.pressure").alias("pressure"),

    col("wind.speed").alias("wind_speed"),
    col("wind.deg").alias("wind_degree"),

    col("visibility"),

    col("clouds.all").alias("cloudiness"),

    col("weather")[0]["main"].alias("weather_main"),
    col("weather")[0]["description"].alias("weather_description"),

    to_timestamp(
        from_unixtime(col("sys.sunrise"))
    ).alias("sunrise"),

    to_timestamp(
        from_unixtime(col("sys.sunset"))
    ).alias("sunset"),

    to_timestamp(
        col("ingestion_timestamp")
    ).alias("ingestion_timestamp")
)


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_silver")

In [0]:
from pyspark.sql.functions import avg

gold_avg_temp = (
    silver_df
    .groupBy("city")
    .agg(
        avg("temperature")
        .alias("avg_temperature")
    )
)

display(gold_avg_temp)

city,avg_temperature
Bengaluru,23.65
Mumbai,30.93
Pune,26.88
Delhi,35.09
Hyderabad,24.82


In [0]:
gold_avg_temp.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_gold_avg_temp")

In [0]:
spark.sql("SHOW TABLES").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|      big_mart_sales|      false|
| default|      weather_bronze|      false|
| default|weather_gold_avg_...|      false|
| default|      weather_silver|      false|
+--------+--------------------+-----------+

